# Topic: SQL: Second Highest Value / Nth Highest Salary

## Definition
Finding a specific ranked value (e.g., 2nd, 3rd, Nth highest) within a dataset while correctly handling duplicate values (ties) and missing data (NULLs). 

## Why Interviewers Ask This
* Tests your understanding of window functions versus traditional aggregations.
* Evaluates your ability to handle edge cases like ties and NULLs.
* Checks if you write scalable code (subqueries get messy for N=5, window functions scale perfectly).

## Core Concepts
* **`DENSE_RANK()`:** Assigns consecutive ranks, even if there are ties.
* **`RANK()`:** Assigns ranks with gaps if there are ties (e.g., 1, 1, 3).
* **Subqueries with `MAX()`:** Useful for exactly the 2nd highest, but scales poorly.
* **`OFFSET` / `LIMIT`:** Skips rows, but fails completely if ties exist at the top rank.

## When to Use
* Anytime the prompt asks for "Nth largest," "Nth most recent," or "Second highest."
* When you need to return multiple rows that share the same Nth rank (e.g., three people tied for 2nd place).

## Advantages
* **`DENSE_RANK()`:** Most robust, handles ties natively, easily scales to any 'N'.
* **Subquery:** Compatible with older SQL versions that lack window functions.

## Limitations
* **`OFFSET` / `LIMIT`:** Will return the wrong result if duplicates exist.
* **Subquery:** Becomes deeply nested and unreadable for finding the 3rd or 4th highest value.

## Common Comparisons
* **`DENSE_RANK()` vs `RANK()`:** `DENSE_RANK()` never skips numbers after a tie (1, 1, 2). `RANK()` skips numbers (1, 1, 3).
* **Window Functions vs `OFFSET`:** Window functions are evaluated logically across the dataset considering values; `OFFSET` just blindly skips physical rows.

## Common Interview Traps
* **Using `LIMIT 1 OFFSET 1` without `DISTINCT`:** If the top two earners make the exact same amount, you just return the highest salary again.
* **Forgetting NULLs:** Not filtering out `NULL` salaries can skew rankings or return blank records.
* **Subquery returning multiple rows:** Using `=` instead of `IN` when an inner subquery accidentally returns multiple tied records.

## SQL Syntax
```sql
WITH RankedData AS (
    SELECT 
        emp_id, 
        salary,
        DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM employees
    WHERE salary IS NOT NULL
)
SELECT emp_id, salary
FROM RankedData
WHERE rnk = 2; -- Change to N for Nth highest
```

## 45-Second Interview Answer
"To find the Nth highest value, my go-to approach is using a Common Table Expression (CTE) with the `DENSE_RANK()` window function. It's the most reliable method for interviews because it gracefully handles ties without skipping ranks—unlike `RANK()`—and avoids the inaccuracy of `OFFSET` when duplicates exist. It's also infinitely more scalable than writing nested `MAX()` subqueries. I'd also ensure to add a `WHERE column IS NOT NULL` clause to handle missing data safely."

---

## Example Questions and Answers

### Q1. Find the third highest revenue among all products.
**Ideal Interview Answer:**
```sql
WITH RankedProducts AS (
    SELECT product_id, revenue,
           DENSE_RANK() OVER(ORDER BY revenue DESC) as rnk
    FROM products
    WHERE revenue IS NOT NULL
)
SELECT product_id, revenue 
FROM RankedProducts 
WHERE rnk = 3;
```
**Common Mistakes:** Using `ORDER BY revenue DESC LIMIT 1 OFFSET 2`. If the top 3 products have the exact same revenue, this returns the wrong absolute revenue amount.
**Likely Follow-up:** "How would you change this if we only wanted to return a single product, even if multiple products tied for 3rd?"

### Q2. Find the second highest salary in each department separately.
**Ideal Interview Answer:**
```sql
WITH DeptRanks AS (
    SELECT emp_id, department, salary,
           DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) as rnk
    FROM employees
    WHERE salary IS NOT NULL
)
SELECT department, emp_id, salary 
FROM DeptRanks 
WHERE rnk = 2;
```
**Common Mistakes:** Forgetting the `PARTITION BY` clause and just grouping by department, or trying to solve this using `GROUP BY` and a `HAVING` clause, which becomes extremely complex for returning non-aggregated columns like `emp_id`.
**Likely Follow-up:** "What if a department only has one employee? How does your query handle that, and how would you modify it to return NULL or a specific string for that department?"

### Q3. Find the employee(s) who earn the second highest salary (return all if there are ties).
**Ideal Interview Answer:**
```sql
WITH SalaryRanks AS (
    SELECT emp_id, name, salary,
           DENSE_RANK() OVER(ORDER BY salary DESC) as rnk
    FROM employees
    WHERE salary IS NOT NULL
)
SELECT emp_id, name, salary 
FROM SalaryRanks 
WHERE rnk = 2;
```
**Common Mistakes:** Attempting to use the subquery approach (`WHERE salary < (SELECT MAX(salary)...)`) but failing to realize it can be slow, or using `LIMIT 1` in the outer query, which cuts off the tied employees. 
**Likely Follow-up:** "If the `employees` table has 100 million rows, how might this window function perform, and are there indexing strategies to help?"

### Q4. Find the second most recent order date for each customer.
**Ideal Interview Answer:**
```sql
WITH OrderRanks AS (
    SELECT customer_id, order_date,
           DENSE_RANK() OVER(PARTITION BY customer_id ORDER BY order_date DESC) as rnk
    FROM orders
    WHERE order_date IS NOT NULL
)
SELECT customer_id, order_date 
FROM OrderRanks 
WHERE rnk = 2;
```
**Common Mistakes:** Using `RANK()` instead of `DENSE_RANK()`. If a customer placed two separate orders on the exact same most recent date, `RANK()` will assign them both rank 1, and the next order date will be rank 3, meaning rank 2 doesn't exist and the customer is incorrectly filtered out.
**Likely Follow-up:** "If a customer placed two orders on their second most recent date, your query returns two rows for them. How would you ensure only one row per customer is returned?"

## Practice Questions

In [1]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('/home/shail/interview-prep/01_SQL/oracle_hr.db')

### Q1: The "No Second Highest" Edge Case

"Your DENSE_RANK() solution for finding the second highest salary is great. However, I have a slightly different business requirement. I want you to return the 2nd highest salary from the employees table. BUT, if there is no 2nd highest salary (for example, if there is only 1 employee in the table, or all employees make the exact same amount), your query must return NULL instead of returning an empty result set."

Table: employees
- emp_id (INT)
- salary (DECIMAL)

(Sample scenario: If the table only has one row (1, 100000), the query should return a single row/column containing NULL. If it has (1, 100000) and (2, 50000), it should return 50000).


Answer: (As the query is not from HR Schema, the solution is below. We can see the same kind of query and output from HR schema belw that):
```sql
SELECT MAX(salary) AS second_highest_salary
FROM (
    SELECT salary, 
           DENSE_RANK() OVER(ORDER BY salary DESC) as rnk
    FROM employees
) ranked_salaries
WHERE rnk = 2;
```

In [11]:
pd.read_sql_query(sql= """
select max(salary) as second_high_salary
from (
    select salary, dense_rank() over(order by salary desc) as sal_rank
    from employees) temp
where sal_rank= 200; -- Using rank = 200 to make sure there are no rows for that and it returns null.
""", con= conn)

,second_high_salary
0,None
